<a href="https://colab.research.google.com/github/EmilioUcelayLojo/AA3/blob/laboratorios_practicas/lab7/lab7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eirasf/GCED-AA3/blob/main/lab7/lab7.ipynb)

# Práctica 7: Autoaprendizaje

## Pre-requisitos

### Instalar paquetes

Si la práctica requiere algún paquete de Python, habrá que incluir una celda en la que se instalen. Si usamos un paquete que se ha utilizado en prácticas anteriores, podríamos dar por supuesto que está instalado pero no cuesta nada satisfacer todas las dependencias en la propia práctica para reducir las dependencias entre ellas.

### NOTA: En <font color='red'>Google Colab</font> hay que instalar los paquetes EN CADA EJECUCIÓN

In [ ]:
# Ejemplo de instalación de tensorflow 2.0
#%tensorflow_version 2.x
# !pip3 install tensorflow  # NECESARIO SOLO SI SE EJECUTA EN LOCAL
import tensorflow as tf

# Hacemos los imports que sean necesarios
import numpy as np

# Autoaprendizaje sobre Fashion-MNIST

Lo primero que tenemos que hacer es cargar el dataset. Tomaremos el 5% de los datos como etiquetados (`x_train` e `y_train`) y el resto como no etiquetados (`unlabeled_train`).

In [ ]:
labeled_data = 0.05 # Vamos a usar el etiquetado de sólo el 5% de los datos
np.random.seed(42)

(x_train, y_train), (x_test, y_test), = tf.keras.datasets.fashion_mnist.load_data()

indexes = np.arange(len(x_train))
np.random.shuffle(indexes)
ntrain_data = int(labeled_data*len(x_train))
unlabeled_train = x_train[indexes[ntrain_data:]]
x_train = x_train[indexes[:ntrain_data]]
y_train = y_train[indexes[:ntrain_data]]

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
# TODO: Haz el preprocesado que necesites aquí (Vuelve sobre esta celda tras estudiar las posteriores)
# Asegúrate de tener los shapes apropiados y de que las variables se representen de una manera que el modelo pueda predecirlas.

# Normalizar los valores de los píxeles al rango [0, 1]
x_train = x_train.astype(np.float32) / 255.0
unlabeled_train = unlabeled_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

# Aplanar las imágenes para que sean vectores de características
x_train = x_train.reshape(x_train.shape[0], -1)
unlabeled_train = unlabeled_train.reshape(unlabeled_train.shape[0], -1)
x_test = x_test.reshape(x_test.shape[0], -1)


## Función de autoaprendizaje

Vamos a crear nuestra propia función de autoaprendizaje. La idea es comenzar aprendiendo un modelo para los datos etiquetados y predecir con ese modelo los datos sin etiquetar. De dichas predicciones, tomaremos aquellas en las que el modelo tiene mayor confianza (cuya probabilidad exceda un umbral `thresh`) y las añadiremos al conjunto de datos etiquetados. Repetiremos el proceso un número predeterminado de veces `train_epochs`.

El pseudocódigo es el siguiente:



**self_training** *(model, x_train, y_train, unlabeled_data, thresh, train_epochs)*

1. $train\_data, train\_label \leftarrow x\_train, y\_train$
1. **Desde** $n = 1 .. train\_epochs$ **hacer**
    1. Instancia un nuevo *model* y entrénalo usando las variables *train\_data* y *train\_label*
    2. $y\_pred \leftarrow model(unlabeled\_data)$
    3. $y\_class, y\_value \leftarrow $ Clase ganadora en *y_pred* con su valor
    4. $train\_data, train\_label \leftarrow x\_train, y\_train$
    5. **Para cada elemento** (x_u, y_c, y_v) **de la tupla** (unlabeled_data, y_class, y_value)
        1. **Si** $y\_v > thresh$ **entonces**
            1. Añadimos $x\_u$ e $y\_c$ a train\_data y train\_label, respectivamente.
4. Devolvemos el modelo


La función asume que `model_creator` es una función sin parámetros que instancia un modelo de Sklearn que, por tanto, cuenta con las funciones [fit](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC.fit), [predict](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC.predict), [predict_proba](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC.predict_proba) y [score](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC.score). Consulta la documentación para identificar cuál te es útil en cada momento.

In [ ]:
# TODO: Implementa el algoritmo self_training tal y como viene en el pseudocódigo.
# TODO: Imprime en cada epoch la precisión y el número de elementos en el conjunto etiquetado,

def self_training(model_creator, x_train, y_train, unlabeled_data, x_test, y_test, thresh=0.5, train_epochs=3):
    train_data = x_train.copy()  # Copiamos los conjuntos de datos para no modificar los originales
    train_label = y_train.copy()
    for n in range(train_epochs):  # Usamos 'n' como variable de iteración
        model = model_creator()
        model.fit(train_data, train_label)  # Entrenamos el modelo con los datos etiquetados actuales
        y_pred_proba = model.predict_proba(unlabeled_data)  # Obtenemos las probabilidades de las predicciones para los datos no etiquetados
        y_pred = np.argmax(y_pred_proba, axis=1)  # Obtenemos las clases predichas
        y_pred_proba_max = np.max(y_pred_proba, axis=1)  # Obtenemos las probabilidades máximas para cada predicción

        # Añadimos las nuevas muestras con alta confianza al conjunto de entrenamiento
        new_train_data = []
        new_train_label = []
        for j in range(len(unlabeled_data)):
            if y_pred_proba_max[j] > thresh:
                new_train_data.append(unlabeled_data[j])
                new_train_label.append(y_pred[j])

        # Agregamos los nuevos datos etiquetados (solo una vez)
        train_data = np.concatenate((train_data, np.array(new_train_data)))
        train_label = np.concatenate((train_label, np.array(new_train_label)))

        # Imprimimos la precisión y el número de elementos en el conjunto etiquetado (usando 'n')
        accuracy = model.score(x_test, y_test)
        print(f"Epoch {n+1}: Precisión = {accuracy:.4f}, Datos etiquetados = {len(train_data)}")

    return model

### Entrenamos nuestro clasificador

Usa lo hecho anteriormente para entrenar tu clasificador de una manera semi-supervisada. Utiliza para ello el [SVM](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) de sklearn (vigila el parámetro probability).

In [ ]:
# Define la función para llamar al SVM
from sklearn.svm import SVC
model_func = lambda: SVC(kernel='linear', probability=True, max_iter=200)

In [ ]:
# TODO: Entrena tu clasificador
svm_model = self_training(model_func, x_train, y_train, unlabeled_train, x_test, y_test, thresh=0.8)  # Usamos self_training con los datos y parámetros adecuados


/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


Epoch 1: Precisión = 0.7980, Datos etiquetados = 30935


/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


Epoch 2: Precisión = 0.7519, Datos etiquetados = 66023


/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


Epoch 3: Precisión = 0.7999, Datos etiquetados = 107321


## Mejorando el código

Tal como hemos visto en clase de teoría, este código puede ser mejorado de varias maneras:

  1. **Asignar más peso a las variables etiquetadas**.
  1. **Asignar un peso en función de la certeza de la predicción en las variables sin etiquetar**.

### TRABAJO: Modifica la función self_training para tener en cuenta todos los puntos mencionados anteriormente

In [ ]:
# TODO: reescribe la función self_training para incorporar las mejoras mencionadas anteriormente

def self_training_v2(model_creator, x_train, y_train, unlabeled_data, x_test, y_test, thresh=0.8, train_epochs=3):
    train_data = x_train.copy()
    train_label = y_train.copy()

    for n in range(train_epochs):
        model = model_creator()

        # Asignamos pesos a las muestras: mayor peso a las etiquetadas
        sample_weight = np.ones(len(train_data))
        sample_weight[:len(x_train)] *= 2  # Doble peso a las etiquetadas

        model.fit(train_data, train_label, sample_weight=sample_weight)

        y_pred_proba = model.predict_proba(unlabeled_data)
        y_pred = np.argmax(y_pred_proba, axis=1)
        y_pred_proba_max = np.max(y_pred_proba, axis=1)

        # Añadimos nuevas muestras con alta confianza y pesos proporcionales a la confianza
        new_train_data = []
        new_train_label = []
        new_sample_weight = []

        for j in range(len(unlabeled_data)):
            if y_pred_proba_max[j] > thresh:
                new_train_data.append(unlabeled_data[j])
                new_train_label.append(y_pred[j])
                new_sample_weight.append(y_pred_proba_max[j])  # Peso proporcional a la confianza

        # Agregamos los nuevos datos etiquetados y sus pesos
        train_data = np.concatenate((train_data, np.array(new_train_data)))
        train_label = np.concatenate((train_label, np.array(new_train_label)))
        sample_weight = np.concatenate((sample_weight, np.array(new_sample_weight)))

        accuracy = model.score(x_test, y_test)
        print(f"Epoch {n+1}: Precisión = {accuracy:.4f}, Datos etiquetados = {len(train_data)}")

    return model


In [ ]:
# TODO: Entrena tu clasificador
svm_model_v2 = self_training_v2(model_func, x_train, y_train, unlabeled_train, x_test, y_test, thresh=0.8)

/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


Epoch 1: Precisión = 0.7909, Datos etiquetados = 30590


/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


Epoch 2: Precisión = 0.7851, Datos etiquetados = 66206


/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


Epoch 3: Precisión = 0.7965, Datos etiquetados = 107375


### Creamos nuestro clasificador en Tensorflow

A continuación crearemos nuestro propio modelo en TensorFlow pero asegurándonos que tiene los métodos de la API de `sklearn` que se utilizan en `self_training`. Ya deberías de ser capaz de realizar este trabajo con muy poca ayuda. El diseño del clasificador es libre (capas densas, convolucionales, ...), puedes crearlo como quieras. Lo único que tenemos que tener en cuenta es que tenemos que encapsular nuestro modelo en una clase cuyas funciones tengan el mismo nombre (y los mismos parámetros) que las existentes en sklearn.

In [ ]:
# TODO: crea tu propio clasificador

class MiClasificador:

    def __init__(self, optimizer):
        # TODO : define el modelo
        # Define el modelo (ejemplo con capas densas)
        self.model = tf.keras.models.Sequential([
            tf.keras.layers.Dense(128, activation='relu', input_shape=(784,)), # Input shape para Fashion-MNIST
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dense(10, activation='softmax')  # 10 clases para Fashion-MNIST
        ])
        # TODO: crea el optimizador
        self.optimizer = optimizer
        # TODO: compila el modelo
        self.model.compile(
            loss='sparse_categorical_crossentropy', # Para clasificación multiclase con etiquetas enteras
            optimizer=self.optimizer,
            metrics=['accuracy']
        )

    def fit(self, X, y, sample_weight=None):
        # TODO: entrena el modelo. Escoge el tamaño de batch y el número de epochs que quieras
        self.model.fit(X, y, epochs=10, batch_size=32, sample_weight=sample_weight) # Ajusta epochs y batch_size

    def predict(self, X):
        # TODO: devuelve la clase ganadora
        return np.argmax(self.model.predict(X), axis=1)

    def predict_proba(self, X):
        # TODO: devuelve las probabilidades de cada clase
        return self.model.predict(X)

    def score(self, X, y):
        # TODO: devuelve el accuracy del clasificador
         _, accuracy = self.model.evaluate(X, y, verbose=0)  # verbose=0 para no mostrar la salida
         return accuracy

    def __del__(self):
        del self.model
        tf.keras.backend.clear_session() # Necesario para liberar la memoria en GPU

### Entrenando el modelo

Crea una función que nos permita crear el modelo en cada iteración.

In [ ]:
# TODO: Entrena el modelo

def model_creator():
    return MiClasificador(tf.keras.optimizers.Adam(learning_rate=0.001))

# Entrenamos el modelo
svm_model_tf = self_training(model_creator, x_train, y_train, unlabeled_train, x_test, y_test, thresh=0.8)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5250 - loss: 1.3712
Epoch 2/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7639 - loss: 0.6635
Epoch 3/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8017 - loss: 0.5577
Epoch 4/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8207 - loss: 0.4949
Epoch 5/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8590 - loss: 0.4275
Epoch 6/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8517 - loss: 0.4066
Epoch 7/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8615 - loss: 0.3818
Epoch 8/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8603 - loss: 0.3802
Epoch 9/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8807 - loss: 0.3281
Epoch 10/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8920 - loss: 0.3067
1782/1782 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
Epoch 1: Precisión = 0.7910, Datos etiquetados = 41728
Epoch 1/10
1304/1304 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accur

# ¡ENHORABUENA! Has completado la práctica de auto-aprendizaje.
